# Register + CPA Feed (Standalone)

- Single notebook, all logic visible
- Local persistence enabled by default
- Supabase is optional


In [6]:
%pip -q install requests urllib3
print("deps ready")


Note: you may need to restart the kernel to use updated packages.
deps ready


In [7]:
# config
CONFIG = {
  # mode: register_and_feed | register_only | feed_existing
  "mode": "register_and_feed",
  "register_total": 1,
  "register_interval_seconds": 1.0,
  "feed_email": "",
  "feed_index": -1,

  # persistence: register | feed_success
  "persist_on": "feed_success",
  "workspace_dir": "/content/keygen_data",
  "accounts_file": "accounts.txt",
  "csv_file": "registered_accounts.csv",
  "result_file": "last_result.json",

  "oauth": {
    "client_id": "app_EMoamEEZ73f0CkXaXp7hrann",
    "redirect_uri": "http://localhost:1455/auth/callback",
    "scope": "openid profile email offline_access"
  },

  "network": {
    "use_proxy": False,
    "proxy_url": "",
    "verify_tls": False,
    "timeout_seconds": 30
  },

  "mail": {
    "worker_domain": "<MAIL_WORKER_DOMAIN>",
    "admin_password": "<MAIL_ADMIN_PASSWORD>",
    "email_domain": "<MAIL_EMAIL_DOMAIN>",
    "cf_token": "",
    "name_prefix": "tmp"
  },

  "otp": {
    "timeout_seconds": 120,
    "poll_interval_seconds": 3,
    "recent_seconds": 180,
    "pre_poll_window_seconds": 30,
    "login_max_attempts": 3,
    "latest_only": True,
    "max_pages": 3,
    "page_size": 10
  },

  "cpa": {
    "enabled": True,
    "base": "<CPA_BASE_URL>",
    "key": "<CPA_KEY>",
    "filename_prefix": "codex-",
    "verify_list": True
  },

  # optional
  "supabase": {
    "enabled": True,
    "url": "<SUPABASE_URL>",
    "key": "<SUPABASE_SERVICE_KEY>",
    "accounts_table": "accounts_pool",
    "runs_table": "keygen_runs"
  }
}

print("config ready")


config ready


## 1) Base Utilities


In [8]:
import base64,csv,json,os,quopri,random,re,secrets,string,time,uuid
from datetime import datetime,timezone
from pathlib import Path
from typing import Any,Dict,List,Tuple
from urllib.parse import urlencode,urlparse,parse_qs
import requests,urllib3
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

AUTH="https://auth.openai.com"
SENTINEL="https://sentinel.openai.com/backend-api/sentinel/req"
UA="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
PROXY_KEYS=["HTTP_PROXY","HTTPS_PROXY","ALL_PROXY","http_proxy","https_proxy","all_proxy"]

def ts(): return datetime.now(timezone.utc).isoformat()
def log(m): print(f"[{datetime.now().strftime('%H:%M:%S')}] {m}")

def jwt_payload(t):
    t=str(t or "").strip()
    if t.count('.')<2:return {}
    b=t.split('.')[1]; b+='='*((4-len(b)%4)%4)
    try:return json.loads(base64.urlsafe_b64decode(b.encode()).decode('utf-8','ignore'))
    except:return {}

def parse_ts(v):
    if isinstance(v,(int,float)):return float(v)
    s=str(v or '').strip()
    if not s:return 0.0
    try:
        if s.endswith('Z'):s=s[:-1]+'+00:00'
        return datetime.fromisoformat(s).timestamp()
    except:return 0.0

def tmo(net): return int(net.get('timeout_seconds',30) or 30)
def tls(net): return bool(net.get('verify_tls',True))

def clear_proxy(net):
    if not bool(net.get('use_proxy',False)):
        for k in PROXY_KEYS: os.environ.pop(k,None)

def sess(net):
    s=requests.Session(); s.trust_env=False
    r=Retry(total=3,connect=3,read=3,backoff_factor=0.8,status_forcelist=[429,500,502,503,504],allowed_methods=frozenset(["GET","POST"]),raise_on_status=False)
    a=HTTPAdapter(max_retries=r); s.mount('http://',a); s.mount('https://',a)
    if bool(net.get('use_proxy',False)) and str(net.get('proxy_url','')).strip():
        p=str(net['proxy_url']).strip(); s.proxies.update({'http':p,'https':p})
    return s

def ws(cfg):
    root=Path(str(cfg.get('workspace_dir','./workspace'))).expanduser(); root.mkdir(parents=True,exist_ok=True)
    af=root/str(cfg.get('accounts_file','accounts.txt')); cf=root/str(cfg.get('csv_file','registered_accounts.csv')); rf=root/str(cfg.get('result_file','last_result.json'))
    if not af.exists(): af.write_text('',encoding='utf-8')
    if not cf.exists(): cf.write_text('email,password,registered_at,mode,status,token_len,cpa_uploaded,error\n',encoding='utf-8')
    return {'root':root,'accounts_file':af,'csv_file':cf,'result_file':rf}

def load_accounts(p):
    out=[]
    if not p.exists(): return out
    for ln in p.read_text(encoding='utf-8',errors='ignore').splitlines():
        if ':' not in ln: continue
        e,pw=ln.split(':',1); e=e.strip(); pw=pw.strip()
        if e and pw: out.append((e,pw))
    return out

def pick(accounts,email,index):
    if not accounts: raise RuntimeError('accounts file is empty')
    e=str(email or '').strip().lower()
    if e:
        for a,b in accounts:
            if a.lower()==e:return a,b
        raise RuntimeError(f'account not found: {e}')
    if index>=0:
        if index>=len(accounts): raise RuntimeError(f'feed_index out of range: {index}')
        return accounts[index]
    return accounts[-1]

def append_unique(p,email,password):
    if email.lower() in {e.lower() for e,_ in load_accounts(p)}: return False
    with p.open('a',encoding='utf-8') as f:f.write(f"{email}:{password}\n")
    return True

def append_csv(p,row):
    fn=['email','password','registered_at','mode','status','token_len','cpa_uploaded','error']
    with p.open('a',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fn); w.writerow({k:row.get(k,'') for k in fn})

def rpass(n=16): return ''.join(random.choice(string.ascii_letters+string.digits+'!@#$%^&*') for _ in range(n))
def rlocal(prefix='tmp'): return prefix+''.join(random.choices(string.ascii_lowercase,k=10))+''.join(random.choices(string.digits,k=2))
def rprofile():
    f=random.choice(['James','Mary','John','Linda','Robert','Sarah']); l=random.choice(['Smith','Johnson','Brown','Wilson']); y=random.randint(1995,2004); m=random.randint(1,12); d=random.randint(1,28)
    return f,l,f"{y:04d}-{m:02d}-{d:02d}"

def pkce():
    v=secrets.token_urlsafe(64)
    import hashlib
    c=base64.urlsafe_b64encode(hashlib.sha256(v.encode()).digest()).decode().rstrip('=')
    return v,c

def create_mail(s,net,mail,name=''):
    wd=str(mail.get('worker_domain','')).strip(); ap=str(mail.get('admin_password','')).strip(); dm=str(mail.get('email_domain','')).strip(); pf=str(mail.get('name_prefix','tmp'))
    if not wd or not ap or not dm: raise RuntimeError('mail worker/admin/domain missing')
    nm=str(name or '').strip() or rlocal(pf)
    r=s.post(f"https://{wd}/admin/new_address",json={'enablePrefix':True,'name':nm,'domain':dm},headers={'x-admin-auth':ap,'Content-Type':'application/json'},timeout=tmo(net),verify=tls(net))
    if r.status_code!=200: raise RuntimeError(f'mail create failed {r.status_code}: {str(r.text or "")[:200]}')
    j=r.json(); return str(j.get('address','')).strip(), str(j.get('jwt','')).strip()

def admin_token(s,net,mail,email):
    wd=str(mail.get('worker_domain','')).strip(); ap=str(mail.get('admin_password','')).strip()
    if not wd or not ap or not email:return ''
    h={'x-admin-auth':ap}
    r1=s.get(f"https://{wd}/admin/address",params={'query':email,'limit':1,'offset':0},headers=h,timeout=tmo(net),verify=tls(net))
    if r1.status_code!=200:return ''
    try:arr=(r1.json() or {}).get('results',[])
    except:arr=[]
    if not arr:return ''
    aid=str((arr[0] or {}).get('id','')).strip();
    if not aid:return ''
    r2=s.get(f"https://{wd}/admin/show_password/{aid}",headers=h,timeout=tmo(net),verify=tls(net))
    if r2.status_code!=200:return ''
    try:return str((r2.json() or {}).get('jwt','')).strip()
    except:return ''

def fetch_mails(s,net,mail,email,cf,max_pages,page_size):
    wd=str(mail.get('worker_domain','')).strip(); tok=str(cf or '').strip() or admin_token(s,net,mail,email)
    if not tok:return []
    rows=[]
    for i in range(max(1,max_pages)):
        r=s.get(f"https://{wd}/api/mails",params={'limit':page_size,'offset':i*page_size},headers={'Authorization':f'Bearer {tok}'},timeout=tmo(net),verify=tls(net))
        if r.status_code!=200:break
        try:batch=(r.json() or {}).get('results',[])
        except:break
        if not isinstance(batch,list) or not batch:break
        rows.extend([x for x in batch if isinstance(x,dict)])
        if len(batch)<page_size:break
    return rows

def mail_ts(m):
    for k in ['createdAt','created_at','created','updatedAt','date','timestamp']:
        if k in m:
            x=parse_ts(m.get(k));
            if x>0:return x
    return 0.0

def chunks(o,out):
    if o is None:return
    if isinstance(o,str):
        if o.strip():out.append(o)
        return
    if isinstance(o,bytes):
        out.append(o.decode('utf-8','ignore')); return
    if isinstance(o,list):
        for x in o:chunks(x,out); return
    if isinstance(o,dict):
        for k in ['subject','raw','rawData','rawdata','text','textBody','plain','html','htmlBody','content','body']:
            if k in o:chunks(o.get(k),out)
        for v in o.values():
            if isinstance(v,(dict,list)):chunks(v,out)

def extract_otp(mail):
    c=[]; chunks(mail,c)
    txt='\n'.join(c)
    try: txt=quopri.decodestring(txt).decode('utf-8','ignore')
    except: pass
    for p in [r'\b(\d{6})\b',r'code\D{0,20}(\d{6})',r'otp\D{0,20}(\d{6})']:
        m=re.search(p,txt,flags=re.I)
        if m:return m.group(1)
    return ''

def wait_otp(s,net,mail,otp,email,cf,not_before_ts=0.0,require_new_mail=False,reject_codes=None):
    to=int(otp.get('timeout_seconds',120) or 120); poll=float(otp.get('poll_interval_seconds',3) or 3); recent=int(otp.get('recent_seconds',180) or 180)
    pre=int(otp.get('pre_poll_window_seconds',30) or 30)
    latest=bool(otp.get('latest_only',True)); mp=int(otp.get('max_pages',3) or 3); ps=int(otp.get('page_size',10) or 10)
    st=time.time()
    nb=float(not_before_ts or 0.0)
    rn=bool(require_new_mail)
    rejects={str(x).strip() for x in (reject_codes or []) if str(x or '').strip()}
    def in_pre(mt):
        if mt<=0:return False
        return mt>=(st-pre) and mt<=(st+10)
    log(f'OTP polling start: timeout={to}s latest_only={latest} recent={recent}s pre={pre}s not_before={int(nb) if nb>0 else 0} require_new={rn} reject={len(rejects)}')
    base=fetch_mails(s,net,mail,email,cf,mp,ps); ids={str(x.get('id')) for x in base if isinstance(x,dict) and x.get('id') is not None}; bts=max([mail_ts(x) for x in base if isinstance(x,dict)] + [0.0])
    log(f'existing mails before polling: {len(base)}')
    if latest and (not rn):
        for m in sorted([x for x in base if isinstance(x,dict)],key=mail_ts,reverse=True)[:5]:
            cd=extract_otp(m); mt=mail_ts(m)
            if not cd: continue
            if cd in rejects: continue
            if nb>0 and mt>0 and mt<nb: continue
            if mt==0 or (time.time()-mt<=recent) or in_pre(mt):
                log(f'OTP found before polling: {cd}')
                return cd
    tried=set(); rd=0
    while time.time()-st<to:
        rd+=1; ms=fetch_mails(s,net,mail,email,cf,mp,ps); log(f'poll {rd}: mail_count={len(ms)}')
        sms=sorted([x for x in ms if isinstance(x,dict)],key=mail_ts,reverse=True)
        if latest:
            cand=([x for x in sms if (str(x.get('id')) not in ids) or (mail_ts(x)>=bts)] if rn else sms[:10])
        else:
            cand=[x for x in sms if str(x.get('id')) not in ids or mail_ts(x)>=bts]
        for m in cand:
            cd=extract_otp(m); mt=mail_ts(m)
            if not cd or cd in tried or cd in rejects: continue
            if nb>0 and mt>0 and mt<nb: continue
            if mt>0 and (time.time()-mt>recent) and (not in_pre(mt)): continue
            tried.add(cd); log(f'OTP candidate: {cd}'); return cd
        time.sleep(poll)
    return ''


## 2) Sentinel + Protocol Register/Login


In [9]:
class Pow:
    def __init__(self,device_id,max_attempts=300000): self.device_id=device_id; self.max_attempts=max_attempts; self.sid=str(uuid.uuid4())
    @staticmethod
    def fnv32(t):
        h=2166136261
        for ch in t:
            h^=ord(ch); h=(h*16777619)&0xffffffff
        h^=h>>16; h=(h*2246822507)&0xffffffff; h^=h>>13; h=(h*3266489909)&0xffffffff; h^=h>>16
        return f"{h&0xffffffff:08x}"
    @staticmethod
    def b64(d): return base64.b64encode(json.dumps(d,separators=(',',':'),ensure_ascii=False).encode()).decode()
    def cfg(self):
        pn=random.uniform(1000,50000)
        return ["1920x1080",datetime.now(timezone.utc).strftime("%a %b %d %Y %H:%M:%S GMT+0000 (UTC)"),4294705152,random.random(),UA,"https://sentinel.openai.com/sentinel/sdk.js",None,None,"en-US","en-US,en",random.random(),"vendorSub−undefined","location","Object",pn,self.sid,"",random.choice([4,8,12,16]),time.time()*1000-pn]
    def req(self):
        c=self.cfg(); c[3]=1; c[9]=round(random.uniform(5,50)); return 'gAAAAAC'+self.b64(c)
    def solve(self,seed,diff):
        st=time.time(); c=self.cfg(); dl=len(diff or '0'); diff=diff or '0'
        for i in range(self.max_attempts):
            c[3]=i; c[9]=round((time.time()-st)*1000); pay=self.b64(c)
            if self.fnv32(seed+pay)[:dl] <= diff: log(f'PoW solved: iter={i+1} cost={time.time()-st:.2f}s'); return 'gAAAAAB'+pay+'~S'
        raise RuntimeError('pow exceeded max attempts')

def sentinel_token(s,net,device,flow):
    pw=Pow(device); body={'p':pw.req(),'id':device,'flow':flow}; h={'Content-Type':'text/plain;charset=UTF-8','Origin':'https://sentinel.openai.com','Referer':'https://sentinel.openai.com/backend-api/sentinel/frame.html','User-Agent':UA}
    r=s.post(SENTINEL,data=json.dumps(body),headers=h,timeout=tmo(net),verify=tls(net))
    if r.status_code!=200: raise RuntimeError(f'sentinel challenge failed {r.status_code}')
    j=r.json(); info=j.get('proofofwork',{}) if isinstance(j,dict) else {}
    seed=str(info.get('seed','') or ''); diff=str(info.get('difficulty','0') or '0'); req=bool(info.get('required',False))
    p=pw.solve(seed,diff) if req and seed else pw.req()
    return json.dumps({'p':p,'t':'','c':str(j.get('token','') or ''),'id':device,'flow':flow})

class Protocol:
    def __init__(self,cfg):
        self.cfg=cfg; self.net=dict(cfg.get('network',{}) or {}); self.mail=dict(cfg.get('mail',{}) or {}); self.otp=dict(cfg.get('otp',{}) or {}); self.oauth=dict(cfg.get('oauth',{}) or {})
        self.s=sess(self.net); self.mail_s=sess(self.net); self.device=str(uuid.uuid4()); self.last_register_otp_by_email={}
    def set_dev(self): self.s.cookies.set('oai-did',self.device,domain='auth.openai.com'); self.s.cookies.set('oai-did',self.device,domain='.auth.openai.com')
    def h(self,ref,flow=''):
        d={'accept':'application/json','accept-language':'en-US,en;q=0.9','content-type':'application/json','origin':AUTH,'referer':ref,'user-agent':UA,'oai-device-id':self.device}
        if flow:d['openai-sentinel-token']=sentinel_token(self.s,self.net,self.device,flow)
        return d
    def nav(self,ref=''):
        d={'accept':'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8','accept-language':'en-US,en;q=0.9','user-agent':UA}
        if ref:d['referer']=ref
        return d
    def auth_url(self,ch,st,signup):
        p={'response_type':'code','client_id':str(self.oauth.get('client_id','')),'redirect_uri':str(self.oauth.get('redirect_uri','')),'scope':str(self.oauth.get('scope','openid profile email offline_access')),'code_challenge':ch,'code_challenge_method':'S256','state':st}
        if signup: p['screen_hint']='signup'; p['prompt']='login'
        return f"{AUTH}/oauth/authorize?{urlencode(p)}"
    def post_continue(self,email,signup):
        ref=f"{AUTH}/create-account" if signup else f"{AUTH}/log-in"; b={'username':{'kind':'email','value':email}}
        if signup:b['screen_hint']='signup'
        return self.s.post(f"{AUTH}/api/accounts/authorize/continue",json=b,headers=self.h(ref,'authorize_continue'),timeout=tmo(self.net),verify=tls(self.net))
    @staticmethod
    def code_from_url(u):
        if not u or 'code=' not in u:return ''
        try:
            code=str((parse_qs(urlparse(u).query).get('code') or [''])[0] or '')
        except:
            return ''
        if (not code) or (' ' in code) or (len(code)<12):
            return ''
        return code
    @staticmethod
    def local_from_err(t):
        m=re.search(r"(https?://localhost[^\s'\"]+)",t or '')
        return m.group(1) if m else ''
    def decode_auth_cookie(self):
        raw=''
        for c in self.s.cookies:
            if c.name=='oai-client-auth-session': raw=str(c.value or ''); break
        if not raw:return {}
        p=raw.split('.')[0]; p+='='*((4-len(p)%4)%4)
        try:return json.loads(base64.urlsafe_b64decode(p.encode()).decode('utf-8','ignore'))
        except:return {}
    def consent_select(self,consent_url):
        d=self.decode_auth_cookie(); ws=d.get('workspaces',[]) if isinstance(d,dict) else []
        if not isinstance(ws,list) or not ws:return ''
        w0=ws[0] if isinstance(ws[0],dict) else {}; wid=str(w0.get('id','') or '')
        if not wid:return ''
        hh=self.h(consent_url)
        wr=self.s.post(f"{AUTH}/api/accounts/workspace/select",json={'workspace_id':wid},headers=hh,timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
        if wr.status_code in (301,302,303,307,308):
            loc=str(wr.headers.get('Location','') or '')
            if loc:return AUTH+loc if loc.startswith('/') else loc
        try:wsj=wr.json()
        except:wsj={}
        ws_next=str((wsj.get('continue_url','') if isinstance(wsj,dict) else '') or '')
        if ws_next: ws_next=AUTH+ws_next if ws_next.startswith('/') else ws_next
        orgs=w0.get('orgs',[]) if isinstance(w0,dict) else []
        if (not orgs) and isinstance(wsj,dict):
            orgs=wsj.get('orgs',[]) if isinstance(wsj.get('orgs',[]),list) else []
            if not orgs and isinstance(wsj.get('data',{}),dict):
                orgs=wsj['data'].get('orgs',[]) if isinstance(wsj['data'].get('orgs',[]),list) else []
        if isinstance(orgs,list) and orgs:
            o0=orgs[0] if isinstance(orgs[0],dict) else {}
            oid=str(o0.get('id','') or '')
            projects=o0.get('projects',[]) if isinstance(o0,dict) else []
            pid=str((projects[0] if isinstance(projects,list) and projects and isinstance(projects[0],dict) else {}).get('id','') or '')
            if oid:
                body={'org_id':oid}
                if pid: body['project_id']=pid
                orr=self.s.post(f"{AUTH}/api/accounts/organization/select",json=body,headers=hh,timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
                if orr.status_code in (301,302,303,307,308):
                    loc=str(orr.headers.get('Location','') or '')
                    if loc:return AUTH+loc if loc.startswith('/') else loc
                try:oj=orr.json()
                except:oj={}
                onext=str((oj.get('continue_url','') if isinstance(oj,dict) else '') or '')
                if onext:return AUTH+onext if onext.startswith('/') else onext
        return ws_next or ''
    def follow_code(self,start):
        if not start:return ''
        c=self.code_from_url(start)
        if c:return c
        u=start; sel=False; consent_round=0
        for _ in range(16):
            try:r=self.s.get(u,headers=self.nav(),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
            except requests.exceptions.ConnectionError as e:
                gu=self.local_from_err(str(e)); return self.code_from_url(gu)
            log(f"follow_code: status={r.status_code} url={str(r.url or '')[:120]}")
            c=self.code_from_url(str(r.url or ''))
            if c:return c
            if r.status_code in (301,302,303,307,308):
                loc=str(r.headers.get('Location','') or '')
                c=self.code_from_url(loc)
                if c:return c
                u=AUTH+loc if loc.startswith('/') else loc
                if not u:break
                continue
            if r.status_code==200:
                body=str(r.text or '')
                lu=self.local_from_err(body)
                c=self.code_from_url(lu)
                if c:return c
                if ('/api/oauth/oauth2/auth' in str(r.url or '')) or ('/api/accounts/consent' in str(r.url or '')):
                    try:
                        rr2=self.s.get(str(r.url or ''),headers=self.nav(),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=True)
                        c=self.code_from_url(str(rr2.url or ''))
                        if c:return c
                        for h2 in (rr2.history or []):
                            c=self.code_from_url(str(h2.headers.get('Location','') or ''))
                            if c:return c
                    except requests.exceptions.ConnectionError as e:
                        gu2=self.local_from_err(str(e)); c=self.code_from_url(gu2)
                        if c:return c
                    except Exception:
                        pass
            if r.status_code==200 and 'consent' in str(r.url or ''):
                consent_round+=1
                nxt=self.consent_select(str(r.url or '')); sel=True
                if nxt:
                    u=nxt
                    continue
                if consent_round<8:
                    time.sleep(0.4)
                    continue
            break
        try:
            rr=self.s.get(start,headers=self.nav(),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=True)
            c=self.code_from_url(str(rr.url or ''))
            if c:return c
            for h in (rr.history or []):
                c=self.code_from_url(str(h.headers.get('Location','') or ''))
                if c:return c
        except requests.exceptions.ConnectionError as e:
            gu=self.local_from_err(str(e)); c=self.code_from_url(gu)
            if c:return c
        except Exception:
            pass
        return ''
    def token(self,code,ver):
        b={'grant_type':'authorization_code','client_id':str(self.oauth.get('client_id','')),'code':code,'redirect_uri':str(self.oauth.get('redirect_uri','')),'code_verifier':ver}
        h={'accept':'application/json','content-type':'application/x-www-form-urlencoded','origin':AUTH,'referer':AUTH,'user-agent':UA}
        r=self.s.post(f"{AUTH}/oauth/token",data=b,headers=h,timeout=tmo(self.net),verify=tls(self.net))
        if r.status_code!=200:return {'ok':False,'error':f'oauth_token_failed_{r.status_code}','body':str(r.text or '')[:300]}
        try:j=r.json()
        except:return {'ok':False,'error':'oauth_token_json_parse_failed'}
        if not str(j.get('access_token','') or ''):return {'ok':False,'error':'oauth_token_missing_access_token'}
        return {'ok':True,'tokens':j}
    def register(self,email,password,cf=''):
        self.set_dev(); ver,ch=pkce(); st=secrets.token_urlsafe(32)
        log('register step 0: oauth authorize')
        r0=self.s.get(self.auth_url(ch,st,True),headers=self.nav(),allow_redirects=True,timeout=tmo(self.net),verify=tls(self.net))
        if r0.status_code not in (200,302):return {'ok':False,'error':f'authorize_failed_{r0.status_code}'}
        log('register step 0b: authorize/continue')
        r1=self.post_continue(email,True)
        if r1.status_code!=200:return {'ok':False,'error':f'authorize_continue_failed_{r1.status_code}','body':str(r1.text or '')[:300]}
        log('register step 1: user/register')
        r2=self.s.post(f"{AUTH}/api/accounts/user/register",json={'username':email,'password':password},headers=self.h(f"{AUTH}/create-account/password",'user_register'),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
        if r2.status_code not in (200,301,302):return {'ok':False,'error':f'user_register_failed_{r2.status_code}','body':str(r2.text or '')[:300]}
        log('register step 2: send email otp')
        self.s.get(f"{AUTH}/api/accounts/email-otp/send",headers=self.nav(f"{AUTH}/create-account/password"),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=True)
        self.s.get(f"{AUTH}/email-verification",headers=self.nav(f"{AUTH}/create-account/password"),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=True)
        log('register step 3: wait otp')
        otp_mark=time.time()
        oc=wait_otp(self.mail_s,self.net,self.mail,self.otp,email,cf,not_before_ts=(otp_mark-15))
        if not oc:return {'ok':False,'error':'otp_not_found'}
        log(f'register step 4: validate otp {oc}')
        r3=self.s.post(f"{AUTH}/api/accounts/email-otp/validate",json={'code':oc},headers=self.h(f"{AUTH}/email-verification"),timeout=tmo(self.net),verify=tls(self.net))
        if r3.status_code==401:
            log('register otp validate 401, retry otp fetch once')
            oc2=wait_otp(self.mail_s,self.net,self.mail,self.otp,email,cf,not_before_ts=(time.time()-15))
            if not oc2:return {'ok':False,'error':'otp_not_found_retry'}
            log(f'register step 4b: validate retry otp {oc2}')
            r3=self.s.post(f"{AUTH}/api/accounts/email-otp/validate",json={'code':oc2},headers=self.h(f"{AUTH}/email-verification"),timeout=tmo(self.net),verify=tls(self.net))
            oc=oc2
        if r3.status_code!=200:return {'ok':False,'error':f'otp_validate_failed_{r3.status_code}','body':str(r3.text or '')[:300]}
        self.last_register_otp_by_email[str(email or '').strip().lower()]=str(oc or '').strip()
        f,l,b=rprofile(); log('register step 5: create_account')
        r4=self.s.post(f"{AUTH}/api/accounts/create_account",json={'name':f'{f} {l}','birthdate':b},headers=self.h(f"{AUTH}/about-you",'create_account'),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
        if r4.status_code not in (200,301,302):
            bd=str(r4.text or '')[:300].lower()
            if 'already_exists' not in bd:return {'ok':False,'error':f'create_account_failed_{r4.status_code}','body':bd}
        return {'ok':True,'email':email,'password':password,'cf_token':cf,'otp':oc,'code_verifier':ver}
    def login(self,email,password,cf=''):
        self.set_dev(); ver,ch=pkce(); st=secrets.token_urlsafe(32)
        log('login step 1: oauth authorize')
        r0=self.s.get(self.auth_url(ch,st,False),headers=self.nav(),allow_redirects=True,timeout=tmo(self.net),verify=tls(self.net))
        if r0.status_code not in (200,302):return {'ok':False,'error':f'authorize_failed_{r0.status_code}'}
        log('login step 2: authorize/continue')
        r1=self.post_continue(email,False)
        if r1.status_code!=200:return {'ok':False,'error':f'authorize_continue_failed_{r1.status_code}','body':str(r1.text or '')[:300]}
        log('login step 3: password/verify')
        r2=self.s.post(f"{AUTH}/api/accounts/password/verify",json={'password':password},headers=self.h(f"{AUTH}/log-in/password",'password_verify'),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
        if r2.status_code!=200:return {'ok':False,'error':f'password_verify_failed_{r2.status_code}','body':str(r2.text or '')[:300]}
        try:j2=r2.json()
        except:j2={}
        pt=str((j2.get('page') or {}).get('type','') or ''); cu=str(j2.get('continue_url','') or '')
        if not cu:return {'ok':False,'error':'continue_url_missing_after_password'}
        if 'email-verification' in cu or pt=='email_otp_verification':
            log('login step 3.5: otp required')
            try:
                self.s.get(f"{AUTH}/api/accounts/email-otp/send",headers=self.nav(f"{AUTH}/log-in/password"),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=True)
                log('login step 3.5a: force send login otp')
            except Exception:
                pass
            otp_mark=time.time()
            rk=str(self.last_register_otp_by_email.get(str(email or '').strip().lower(),'') or '').strip()
            reject=set([rk]) if rk else set()
            max_try=max(1,int(self.otp.get('login_max_attempts',3) or 3))
            r3=None
            for i in range(max_try):
                oc=wait_otp(self.mail_s,self.net,self.mail,self.otp,email,cf,not_before_ts=otp_mark,require_new_mail=True,reject_codes=reject)
                if not oc:
                    continue
                r3=self.s.post(f"{AUTH}/api/accounts/email-otp/validate",json={'code':oc},headers=self.h(f"{AUTH}/email-verification"),timeout=tmo(self.net),verify=tls(self.net))
                if r3.status_code==200:
                    break
                reject.add(str(oc or '').strip())
                log(f'login otp validate failed status={r3.status_code} attempt={i+1}/{max_try}')
            if (r3 is None) or (r3.status_code!=200):
                stc=(0 if r3 is None else r3.status_code)
                body=('' if r3 is None else str(r3.text or '')[:300])
                return {'ok':False,'error':f'otp_validate_login_failed_{stc}','body':body}
            try:cu=str((r3.json() or {}).get('continue_url','') or cu)
            except:pass
        log('login step 4: consent flow for auth code')
        consent_url=(AUTH+cu) if str(cu or '').startswith('/') else str(cu or '')
        cd=''
        try:
            r4=self.s.get(consent_url,headers=self.nav(),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
            log(f"consent get: status={r4.status_code} url={str(getattr(r4,'url','') or '')[:120]}")
            if r4.status_code in (301,302,303,307,308):
                loc=str(r4.headers.get('Location','') or '')
                cd=self.code_from_url(loc) or self.follow_code((AUTH+loc) if loc.startswith('/') else loc)
            elif r4.status_code==200:
                cd=self.code_from_url(str(getattr(r4,'url','') or ''))
        except requests.exceptions.ConnectionError as e:
            cd=self.code_from_url(self.local_from_err(str(e)))
        except Exception:
            pass
        if not cd:
            sd=self.decode_auth_cookie()
            ws=sd.get('workspaces',[]) if isinstance(sd,dict) else []
            wid=''
            if isinstance(ws,list) and ws and isinstance(ws[0],dict):
                wid=str(ws[0].get('id','') or '')
            if wid:
                hh=self.h(consent_url)
                try:
                    wsr=self.s.post(f"{AUTH}/api/accounts/workspace/select",json={'workspace_id':wid},headers=hh,timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
                    log(f"workspace/select: status={wsr.status_code}")
                    if wsr.status_code in (301,302,303,307,308):
                        loc=str(wsr.headers.get('Location','') or '')
                        cd=self.code_from_url(loc)
                    if not cd and wsr.status_code==200:
                        try:wsj=wsr.json()
                        except:wsj={}
                        wnext=str((wsj.get('continue_url','') if isinstance(wsj,dict) else '') or '')
                        orgs=[]
                        if isinstance(wsj,dict) and isinstance(wsj.get('data',{}),dict):
                            orgs=wsj['data'].get('orgs',[]) if isinstance(wsj['data'].get('orgs',[]),list) else []
                        if (not orgs) and isinstance(wsj,dict):
                            orgs=wsj.get('orgs',[]) if isinstance(wsj.get('orgs',[]),list) else []
                        if isinstance(orgs,list) and orgs:
                            o0=orgs[0] if isinstance(orgs[0],dict) else {}
                            oid=str(o0.get('id','') or '')
                            projects=o0.get('projects',[]) if isinstance(o0,dict) else []
                            pid=str((projects[0] if isinstance(projects,list) and projects and isinstance(projects[0],dict) else {}).get('id','') or '')
                            if oid:
                                body={'org_id':oid}
                                if pid: body['project_id']=pid
                                orr=self.s.post(f"{AUTH}/api/accounts/organization/select",json=body,headers=hh,timeout=tmo(self.net),verify=tls(self.net),allow_redirects=False)
                                log(f"organization/select: status={orr.status_code}")
                                if orr.status_code in (301,302,303,307,308):
                                    loc=str(orr.headers.get('Location','') or '')
                                    cd=self.code_from_url(loc)
                                if (not cd) and orr.status_code==200:
                                    try:oj=orr.json()
                                    except:oj={}
                                    onext=str((oj.get('continue_url','') if isinstance(oj,dict) else '') or '')
                                    if onext: cd=self.follow_code((AUTH+onext) if onext.startswith('/') else onext)
                        if (not cd) and wnext:
                            cd=self.follow_code((AUTH+wnext) if wnext.startswith('/') else wnext)
                except Exception:
                    pass
        if not cd:
            try:
                rf=self.s.get(consent_url,headers=self.nav(),timeout=tmo(self.net),verify=tls(self.net),allow_redirects=True)
                cd=self.code_from_url(str(getattr(rf,'url','') or ''))
                if not cd:
                    for h in (rf.history or []):
                        cd=self.code_from_url(str(h.headers.get('Location','') or ''))
                        if cd: break
            except requests.exceptions.ConnectionError as e:
                cd=self.code_from_url(self.local_from_err(str(e)))
            except Exception:
                pass
        if not cd:return {'ok':False,'error':'authorization_code_not_found'}
        log(f"login step 5: oauth/token code_len={len(str(cd or ''))}")
        tk=self.token(cd,ver)
        if not tk.get('ok'):return tk
        at=str((tk.get('tokens') or {}).get('access_token','') or ''); pl=jwt_payload(at); ns=pl.get('https://api.openai.com/auth',{}) if isinstance(pl,dict) else {}
        aid=str(ns.get('chatgpt_account_id','') or '') if isinstance(ns,dict) else ''
        return {'ok':True,'tokens':tk['tokens'],'account_id':aid,'token_len':len(at)}



## 3) CPA + Supabase Helpers


In [10]:
def cpa_payload(email,tokens):
    at=str(tokens.get('access_token','') or ''); rt=str(tokens.get('refresh_token','') or ''); it=str(tokens.get('id_token','') or '')
    pl=jwt_payload(at); ns=pl.get('https://api.openai.com/auth',{}) if isinstance(pl,dict) else {}
    aid=str(ns.get('chatgpt_account_id','') or '') if isinstance(ns,dict) else ''
    ex=''; ev=pl.get('exp') if isinstance(pl,dict) else None
    if isinstance(ev,(int,float)) and int(ev)>0: ex=datetime.fromtimestamp(int(ev),tz=timezone.utc).isoformat()
    return {'type':'codex','email':email,'expired':ex,'id_token':it,'account_id':aid,'access_token':at,'last_refresh':ts(),'refresh_token':rt}

def cpa_headers(k):
    h={'Content-Type':'application/json'}; k=str(k or '').strip()
    if k: h['Authorization']=f'Bearer {k}'; h['X-Management-Key']=k
    return h

def cpa_safe_name(email):
    e=''.join(ch if (ch.isalnum() or ch in ('-','_','.')) else '_' for ch in str(email or '').strip().lower())
    return e or 'unknown'

def cpa_upload(base,key,name,payload):
    b=str(base or '').strip().rstrip('/')
    if not b:return {'ok':False,'skipped':True,'reason':'cpa_base_empty'}
    n=str(name or '').strip()
    if n and (not n.endswith('.json')): n=f"{n}.json"
    txt=json.dumps(payload,ensure_ascii=False)
    u0=f"{b}/v0/management/auth-files?{urlencode({'name':n})}"; r=requests.post(u0,headers=cpa_headers(key),data=txt,timeout=45)
    body=str(r.text or '')[:800]
    ok=(r.status_code==200)
    if ok:
        try:j=r.json()
        except:j={}
        if isinstance(j,dict) and str(j.get('status','')).lower() not in ('ok','success'):
            ok=False
    return {'ok':ok,'status':r.status_code,'url':u0,'body':body,'name':n}

def cpa_list(base,key):
    b=str(base or '').strip().rstrip('/')
    if not b:return {'ok':False,'skipped':True,'reason':'cpa_base_empty'}
    r=requests.get(f"{b}/v0/management/auth-files",headers=cpa_headers(key),timeout=30)
    try:p=r.json()
    except:p={}
    return {'ok':r.status_code==200,'status':r.status_code,'payload':p}
class Supabase:
    def __init__(self,cfg):
        self.cfg=dict(cfg or {}); self.enabled=bool(self.cfg.get('enabled',False)); self.url=str(self.cfg.get('url','') or '').strip().rstrip('/'); self.key=str(self.cfg.get('key','') or '').strip(); self.acc=str(self.cfg.get('accounts_table','accounts_pool') or 'accounts_pool'); self.runs=str(self.cfg.get('runs_table','keygen_runs') or 'keygen_runs'); self.active=self.enabled and bool(self.url and self.key)
    def h(self,p='return=minimal'): return {'apikey':self.key,'Authorization':f'Bearer {self.key}','Content-Type':'application/json','Prefer':p}
    def upsert(self,item):
        if not self.active:return {'ok':False,'skipped':True,'reason':'supabase_disabled'}
        r=requests.post(f"{self.url}/rest/v1/{self.acc}",params={'on_conflict':'email'},headers=self.h('resolution=merge-duplicates,return=minimal'),data=json.dumps([item],ensure_ascii=False),timeout=45)
        return {'ok':r.status_code in (200,201,204),'status':r.status_code,'body':str(r.text or '')[:300]}
    def insert_run(self,item):
        if not self.active:return {'ok':False,'skipped':True,'reason':'supabase_disabled'}
        r=requests.post(f"{self.url}/rest/v1/{self.runs}",headers=self.h('return=minimal'),data=json.dumps([item],ensure_ascii=False),timeout=45)
        return {'ok':r.status_code in (200,201,204),'status':r.status_code,'body':str(r.text or '')[:300]}



## 4) Supabase Connection Test (Optional)


In [11]:
def test_supabase_connection(cfg):
    sb=Supabase(dict((cfg or {}).get('supabase',{}) or {}))
    if not sb.active:
        return {'ok':False,'skipped':True,'reason':'supabase_disabled_or_missing_url_key'}
    out={'ok':False,'supabase_url':sb.url,'accounts_table':sb.acc,'runs_table':sb.runs}
    try:
        r1=requests.get(f"{sb.url}/rest/v1/{sb.runs}",params={'select':'id','order':'id.desc','limit':1},headers=sb.h(),timeout=30)
        r2=requests.get(f"{sb.url}/rest/v1/{sb.acc}",params={'select':'email','limit':1},headers=sb.h(),timeout=30)
        out['runs_probe']={'status':r1.status_code,'body':str(r1.text or '')[:300]}
        out['accounts_probe']={'status':r2.status_code,'body':str(r2.text or '')[:300]}
        out['ok']=r1.status_code==200 and r2.status_code==200
        if not out['ok']:
            out['error']='probe_failed'
        return out
    except Exception as e:
        out['error']=str(e)
        return out

sb_test=test_supabase_connection(CONFIG)
print(json.dumps(sb_test,ensure_ascii=False,indent=2))


{
  "ok": true,
  "supabase_url": "<SUPABASE_URL>",
  "accounts_table": "accounts_pool",
  "runs_table": "keygen_runs",
  "runs_probe": {
    "status": 200,
    "body": "[]"
  },
  "accounts_probe": {
    "status": 200,
    "body": "[]"
  }
}


## 5) Pipeline Orchestration


In [12]:
def run_pipeline(cfg):
    st=datetime.now(timezone.utc); net=dict(cfg.get('network',{}) or {})
    if not tls(net): urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    clear_proxy(net); w=ws(cfg); sink=Supabase(dict(cfg.get('supabase',{}) or {}))
    mode=str(cfg.get('mode','register_and_feed') or 'register_and_feed').strip(); n=max(0,int(cfg.get('register_total',1) or 1)); gap=float(cfg.get('register_interval_seconds',1.0) or 1.0); po=str(cfg.get('persist_on','feed_success') or 'feed_success').strip()
    cpa=dict(cfg.get('cpa',{}) or {}); cen=bool(cpa.get('enabled',True)); cb=str(cpa.get('base','') or ''); ck=str(cpa.get('key','') or ''); cp=str(cpa.get('filename_prefix','codex-') or 'codex-')
    ok=[]; fail=[]
    def persist(email,password,status,token_len,cpa_ok,err,tokens=None):
        append_unique(w['accounts_file'],email,password)
        append_csv(w['csv_file'],{'email':email,'password':password,'registered_at':ts(),'mode':mode,'status':status,'token_len':token_len,'cpa_uploaded':int(bool(cpa_ok)),'error':err})
        tk=tokens if isinstance(tokens,dict) else {}
        at=str(tk.get('access_token','') or '').strip(); rt=str(tk.get('refresh_token','') or '').strip(); it=str(tk.get('id_token','') or '').strip()
        pl=jwt_payload(at) if at else {}; ev=pl.get('exp') if isinstance(pl,dict) else None
        ex=(datetime.fromtimestamp(int(ev),tz=timezone.utc).isoformat() if isinstance(ev,(int,float)) and int(ev)>0 else None)
        now_iso=ts()
        sink.upsert({'email':email,'password':password,'status':status,'token_len':token_len,'cpa_filename':f"{cp}{cpa_safe_name(email)}.json",'error':err,'access_token':(at or None),'refresh_token':(rt or None),'id_token':(it or None),'token_alive':bool(at),'token_check_method':('oauth_token_exchange' if at else None),'token_expired_at':ex,'token_last_refresh':(now_iso if rt else None),'last_token_check':now_iso,'updated_at':now_iso})
    def reg_and_feed():
        mail=dict(cfg.get('mail',{}) or {}); boot=sess(net); cf=str(mail.get('cf_token','') or '').strip(); email,ct=create_mail(boot,net,mail)
        log(f'register mail created: {email}')
        if not cf: cf=ct or admin_token(boot,net,mail,email)
        pwd=rpass(); cli=Protocol(cfg); rg=cli.register(email,pwd,cf)
        if not rg.get('ok'): return {'ok':False,'email':email,'password':pwd,'stage':'register','error':str(rg.get('error','register_failed')),'debug':rg}
        if mode=='register_only': persist(email,pwd,'registered_only',0,False,'',None); return {'ok':True,'email':email,'password':pwd,'stage':'register_only','token_len':0,'cpa':{'ok':False,'skipped':True,'reason':'register_only'}}
        lg=cli.login(email,pwd,cf)
        if not lg.get('ok'):
            if po=='register': persist(email,pwd,'registered_login_failed',0,False,str(lg.get('error','')),None)
            return {'ok':False,'email':email,'password':pwd,'stage':'login','error':str(lg.get('error','login_failed')),'debug':lg}
        tk=lg.get('tokens',{}) if isinstance(lg.get('tokens'),dict) else {}; tl=len(str(tk.get('access_token','') or '')); name=f"{cp}{cpa_safe_name(email)}"; cr={'ok':False,'skipped':True,'reason':'cpa_disabled'}
        if cen: cr=cpa_upload(cb,ck,name,cpa_payload(email,tk))
        save=(po=='register') or (po=='feed_success' and bool(cr.get('ok') or not cen))
        if save:
            stt='fed' if bool(cr.get('ok')) else ('token_ready' if not cen else 'feed_failed')
            persist(email,pwd,stt,tl,bool(cr.get('ok')),'' if bool(cr.get('ok')) else str(cr.get('body','')),tk)
        return {'ok':bool(cr.get('ok') or not cen),'email':email,'password':pwd,'stage':'fed' if bool(cr.get('ok')) else ('token_ready' if not cen else 'feed_failed'),'token_len':tl,'account_id':str(lg.get('account_id','') or ''),'cpa':cr}
    def feed_existing():
        email,pwd=pick(load_accounts(w['accounts_file']),str(cfg.get('feed_email','') or ''),int(cfg.get('feed_index',-1) or -1)); mail=dict(cfg.get('mail',{}) or {}); boot=sess(net); cf=str(mail.get('cf_token','') or '').strip() or admin_token(boot,net,mail,email)
        cli=Protocol(cfg); lg=cli.login(email,pwd,cf)
        if not lg.get('ok'): return {'ok':False,'email':email,'stage':'login_existing','error':str(lg.get('error','login_failed')),'debug':lg}
        tk=lg.get('tokens',{}) if isinstance(lg.get('tokens'),dict) else {}; tl=len(str(tk.get('access_token','') or '')); name=f"{cp}{cpa_safe_name(email)}"; cr={'ok':False,'skipped':True,'reason':'cpa_disabled'}
        if cen: cr=cpa_upload(cb,ck,name,cpa_payload(email,tk))
        if bool(cr.get('ok')) or not cen: persist(email,pwd,'fed_existing' if cen else 'token_ready_existing',tl,bool(cr.get('ok')),'',tk)
        return {'ok':bool(cr.get('ok') or not cen),'email':email,'stage':'feed_existing','token_len':tl,'account_id':str(lg.get('account_id','') or ''),'cpa':cr}
    if mode in ('register_and_feed','register_only'):
        for i in range(n):
            log(f'run register item {i+1}/{n}')
            try:item=reg_and_feed()
            except Exception as e:item={'ok':False,'error':str(e),'stage':'exception'}
            (ok if item.get('ok') else fail).append(item)
            if i<n-1 and gap>0: time.sleep(gap)
    elif mode=='feed_existing':
        try:item=feed_existing()
        except Exception as e:item={'ok':False,'error':str(e),'stage':'exception'}
        (ok if item.get('ok') else fail).append(item)
    else: raise RuntimeError(f'unsupported mode: {mode}')
    vr={'ok':False,'skipped':True,'reason':'cpa_verify_disabled'}
    if bool(cpa.get('verify_list',True)) and cen: vr=cpa_list(cb,ck)
    fi=datetime.now(timezone.utc)
    run={'run_started_at':st.isoformat(),'run_finished_at':fi.isoformat(),'mode':mode,'register_total':n,'feed_ok':len(ok),'feed_fail':len(fail),'verify_found':len((vr.get('payload') or {}).get('data',[])) if isinstance(vr.get('payload'),dict) else 0,'result_json':json.dumps({'ok_items':ok,'fail_items':fail},ensure_ascii=False)}
    sr=sink.insert_run(run)
    res={'ok':len(fail)==0,'mode':mode,'register_total':n,'feed_ok':len(ok),'feed_fail':len(fail),'ok_items':ok,'fail_items':fail,'verify':vr,'supabase_enabled':sink.active,'supabase_run_insert':sr,'workspace':{'root':str(w['root']),'accounts_file':str(w['accounts_file']),'csv_file':str(w['csv_file']),'result_file':str(w['result_file'])},'finished_at':fi.isoformat()}
    w['result_file'].write_text(json.dumps(res,ensure_ascii=False,indent=2),encoding='utf-8')
    return res


## 6) Run


In [23]:
try:
  result=run_pipeline(CONFIG)
except Exception as e:
  result={"ok":False,"error":str(e)}
print(json.dumps(result,ensure_ascii=False,indent=2))


[17:36:15] run register item 1/1


KeyboardInterrupt: 

## 7) Optional Supabase SQL

Use `supabase/schema.sql` from this repo directly. It already matches this notebook, including token fields (`access_token`, `refresh_token`, `id_token`, `token_alive`, `token_check_method`, `token_expired_at`, `token_last_refresh`, `last_token_check`).
